# 02 From-scratch DPO: HH-RLHF and UltraFeedback (Task 4)

In [ ]:
REPO_URL = "https://github.com/NifraWahaj/cs5323_PA2.git"
BRANCH = "main"

import os, subprocess, sys
REPO = "/kaggle/working/sprout"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs visible:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, "Set Accelerator to GPU T4 x2"

In [ ]:
from sproutlm.kaggle_paths import find_run
SFT_BEST = find_run("sft", "best")
print("reference + initial policy:", SFT_BEST)
!cat {SFT_BEST}/sprout_note.json
RUNS = ["hh", "uf"]
import os

In [ ]:
if "hh" in RUNS:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TOKENIZERS_PARALLELISM=false python -m sproutlm.dpo_loop --config configs/dpo_hh.yaml \
        sft_checkpoint={SFT_BEST} output_dir=/kaggle/working/outputs/dpo_hh
    assert os.path.exists("/kaggle/working/outputs/dpo_hh/best/sprout_note.json"), "DPO-HH FAILED - see log above"

In [ ]:
if "uf" in RUNS:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TOKENIZERS_PARALLELISM=false python -m sproutlm.dpo_loop --config configs/dpo_uf.yaml \
        sft_checkpoint={SFT_BEST} output_dir=/kaggle/working/outputs/dpo_uf
    assert os.path.exists("/kaggle/working/outputs/dpo_uf/best/sprout_note.json"), "DPO-UF FAILED - see log above"

In [ ]:
!pip install -q matplotlib
import os
runs = " ".join(f"--run /kaggle/working/outputs/dpo_{r}" for r in RUNS)
!python -m sproutlm.plots {runs} --out /kaggle/working/outputs/plots
from IPython.display import Image, display
for r in RUNS:
    display(Image(f"/kaggle/working/outputs/plots/dpo_{r}.png"))

In [ ]:
!for r in dpo_hh dpo_uf; do echo $r; cat /kaggle/working/outputs/$r/best/sprout_note.json 2>/dev/null; echo; done

In [ ]:
!cd /kaggle/working && zip -qr dpo_run_outputs.zip outputs/dpo_*/metrics.jsonl outputs/dpo_*/config_used.yaml outputs/plots && ls -lh /kaggle/working && du -sh /kaggle/working/outputs/*